## 第11章 文本输入输出和上下文管理

### 1.输入输出

- **标准流**：
    - `sys.stdin`：标准输入流，从键盘读取输入。
    - `sys.stdout`：标准输出流，将输出打印到屏幕。
    - `sys.stderr`：标准错误流，将错误信息打印到屏幕。

- **print()函数**：`print(对象, file=sys.stdout, sep=' ', end='\n', flush=False)`。
    - `对象`：要打印的对象，可以是任意类型。
    - `file`：输出流，默认为标准输出流`sys.stdout`。
    - `sep`：对象之间的分隔符，默认为空格。
    - `end`：对象打印完成后添加的字符串，默认为换行符。
    - `flush`：是否立即刷新输出缓冲区，默认为False。
    - 💡 当输出只是"值用分隔符拼接"时，`sep=`比f-string更简洁高效；需格式化某值时再混用f-string。


In [ ]:
import time
import sys
print('Downloading', file=sys.stdout, sep='', end='')  # end=''，不换行
for n in range(20):
    print('.', end='', flush=True)  # flush=True，立即刷新输出缓冲区
    time.sleep(0.1)
print('\nDone!')

In [ ]:
nearby_properties = {
    "N. Anywhere Ave.":
    {
        123: 156_852,
        124: 157_923,
        126: 163_812,
        127: 144_121,
        128: 166_356,
    },
    "N. Everywhere St.":
    {
        4567: 175_753,
        4568: 166_212,
        4569: 185_123,
    }
}
for street, properties in nearby_properties.items():
    for address, value in properties.items():
        print(street, address, f'{value:,}', sep='\t')  # 一次输出多个值，sep指定分隔符

### 2.文件操作

- **文件打开**：`open(filepath, mode, encoding)`函数。
    - `filepath`：文件路径，可以是相对路径或绝对路径。
    - `mode`：打开模式，可选值有`r`、`w`、`a`、`b`等，默认为`r`。最常见的模式是`r+`，即不擦除文件内容的读写模式。
    - `encoding`：文件编码，默认为`utf-8`。
    <p><img src="img/002.jpg" width="650"></p>

- **文件读取**：
    - `read()`方法：读取文件所有内容，返回一个字符串。
    - `readline()`方法：读取文件一行，返回一个字符串。
    - `readlines()`方法：读取文件所有行，返回一个列表，每个元素为一行字符串。

- **文件流位置**：
    - `tell()`方法：返回当前文件流位置。
    - `seek()`方法：设置文件流位置。
        - `seek(0)`：将文件流位置设置为文件开头。
        - `seek(0, 2)`：将文件流位置设置为文件结尾。

- **文件写入**：注意，**文件写入时永远是覆盖，而不是插入！** 除追加模式外，重写文件需：先读入内存 → 修改 → 再写回。
    - `write()`方法：从当前流位置开始写入字符串，**会覆盖位置处的原有数据**。
    - `writelines()`方法：写入字符串列表到文件，**不会自动添加换行符**，需确保列表元素自带`\n`。
    - `truncate()`方法：从当前流位置截断文件之后的所有内容。
    - `flush()`方法：刷新文件缓冲区，将所有写入内容立即写入文件。

### 3.上下文管理器

- **`with`语句**：自动管理资源，确保资源在使用后正确关闭。`with`语句 = 自动化的`try/finally`语句。

    ```python
    # with复合语句类似以下代码
    VAR = EXPR
    VAR.__enter__()
    try:
        BLOCK
    finally:
        VAR.__exit__()

- **上下文管理器**：一个对象要想成为上下文管理器，就必须实现两个特殊方法：`__enter__()`和`__exit__()`。
    - `__enter__(self)`：在`with`语句执行前调用，返回上下文管理器实例。
    - `__exit__(self, exc_type, exc_val, exc_tb)`：在`with`语句执行后调用，用于清理资源。
        - 可以接收3个参数：异常类型、异常值、异常跟踪。
        - 如果没有异常发生，异常类型为`None`，异常值为`None`，异常跟踪为`None`。
        - 返回值为`False`=异常继续传播，`True`=抑制异常。

In [ ]:
class House:
    def __init__(self, address, key, **rooms):
        self.address = address
        self.__key = key
        self.__locked = True
        self._rooms = {}
        for room, desc in rooms.items():
            self._rooms[room.replace('_', ' ').lower()] = desc

    def unlock(self, key):
        if key == self.__key:
            self.__locked = False
            print('房子解锁')
        else:
            raise RuntimeError('无效钥匙，无法解锁房子')

    def lock(self):
        self.__locked = True
        print('房子上锁')

    def explore(self, room):
        if self.__locked:
            raise RuntimeError('房子已锁，无法看房')
        try:
            return f'房间：{room.lower()}，描述：{self._rooms[room.lower()]}'
        except KeyError as e:
            raise KeyError(f'房间：{room}，不存在') from e

# 自定义上下文管理器
class HouseShowing:
    def __init__(self, house, key):
        self.house = house
        self.key = key

    # 上下文管理器进入前执行
    def __enter__(self):
        self.house.unlock(self.key)  # 解锁
        return self

    # 上下文管理器退出时执行
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            print(f'发生异常：{exc_type.__name__} {exc_val} {exc_tb}')
        self.house.lock()  # 上锁

    def show(self, room):
        print(self.house.explore(room))

house = House("123 Anywhere Street", key=1803, living_room="spacious",office="bright",bedroom="cozy",bathroom="small",kitchen="modern")
with HouseShowing(house, 1803) as showing:
    showing.show("Living Room")
    showing.show("bedroom")
    showing.show("kitchen")

### 4.路径

- **路径模块**：Python提供了两个模块：`os`模块和`pathlib`模块。出于可维护性、可读性以及性能的考虑，建议优先使用pathlib模块。

- **路径组成**：`C:\Windows\System\python37.dll`，`/usr/lib/x86_64-linux-gnu/libpython3.7m.so.1`，这两个路径组成如下：

| 属性          | 说明               | Windows示例               | POSIX示例                   |
| ------------- | ------------------ | ------------------------- | --------------------------- |
| `drive`       | 驱动器             | `C:`                      | `''`（空）                  |
| `root`        | 根目录             | `\`                       | `/`                         |
| `anchor`      | 锚点（drive+root） | `C:\`                     | `/`                         |
| `parent`      | 父目录             | `C:\Windows\System`       | `/usr/lib/x86_64-linux-gnu` |
| `parents[i]`  | 父目录序列         | `[0]:C:\Windows\System`等 | `[0]:/usr/lib`等            |
| `name`        | 最终文件名         | `python37.dll`            | `libpython3.7m.so.1`        |
| `suffix`      | 最后一个后缀       | `.dll`                    | `.1`                        |
| `suffixes[i]` | 所有后缀列表       | `[0]:.dll`                | `[0]:.7m [1]:.so [2]:.1`    |
| `stem`        | 文件名去掉后缀     | `python37`                | `libpython3.7m.so`          |

- **路径拼接：**
    - `joinpath()`方法：`Path.joinpath(PosixPath.home(), '.bash_history')`。
    - `/`运算符：`Path.home() / '.bash_history'`。

- **相对路径：**
    - 相对路径基于当前工作目录(调用脚本的目录)，而非脚本所在目录。
    - 获取当前目录：`Path.cwd()`。
    - 转换为绝对路径：`Path.cwd().resolve()`。
    - 基于包的路径：`Path(__file__).resolve().parents[1]`。


- **异地写入文件：**
    - 文件写入过程中程序崩溃会损坏原文件，安全方案：**先写临时文件，再原子替换**。

In [ ]:
from pathlib import Path

# 读取文件内容
path = Path('res/test.txt')
with path.open('r') as f:
    contents = f.read()
    contents = contents.replace('Tiny', 'Cozy')

# 写入临时文件
tmp_path = path.with_name(path.name + '.tmp')
with tmp_path.open('w') as f:
    f.write(contents)

# 原子替换（系统级操作，极快；崩溃仅丢失临时文件）
tmp_path.replace(path)

### 5.本章小结

- **核心知识脉络**

```text
文本IO与上下文管理器
│
├── 标准流
│   ├── sys.stdin/stdout/stderr
│   ├── print() → stdout（默认），file=sys.stderr → stderr
│   └── sep=/end=/flush= 参数
│
├── 流
│   ├── 文本流 vs 二进制流（本章仅文本流）
│   ├── open() → TextIOWrapper
│   └── 文件模式：r/w/a/x 及 r+/w+/a+/x+
│       ├── ⚠️ w/w+ 截断内容
│       ├── ⚠️ x/x+ 文件存在报错
│       └── readable()/writeable()/seekable() 检查能力
│
├── 读取文件
│   ├── read() → 全部字符串（size=限制字符数）
│   ├── readline() → 单行字符串
│   ├── readlines() → 字符串列表
│   └── for line in house → ⭐ 最Pythonic
│   └── tell()/seek() 流位置控制
│
├── 写入文件
│   ├── write() → ⚠️ 覆盖而非插入，短于旧内容需truncate()
│   ├── writelines() → ⚠️ 不自动加\n
│   ├── print(file=) → ⭐ 利用自动换行和sep=
│   └── truncate() → 截断当前位置后内容
│   └── 换行符：Python自动处理\r\n vs \n，永远只需\n
│
├── 上下文管理器 ⭐⭐
│   ├── with语句 = 自动try/finally
│   ├── 协议：__enter__() + __exit__()
│   │   ├── __enter__() → 返回值赋给as变量
│   │   └── __exit__(exc_type, exc_val, exc_tb) → False=传播，True=抑制⚠️
│   ├── 自定义：类版本（复杂场景）vs 生成器版本（简单场景⭐）
│   │   └── @contextmanager + yield一次 ⚠️
│   ├── 多个上下文：with A() as a, B() as b
│   └── contextlib：suppress/closing/redirect_stdout/ExitStack
│
├── pathlib ⭐⭐
│   ├── 纯路径（PurePath）vs 具体路径（Path）
│   ├── 路径组件：drive/root/anchor/parent/name/stem/suffix
│   ├── 拼接：/ 运算符 ⭐ > joinpath()
│   ├── resolve() → 绝对路径
│   ├── __file__ + resolve() 避免工作目录陷阱 ⭐⭐
│   ├── 操作：mkdir/rename/replace/rmdir/unlink/glob/iterdir/touch
│   ├── 信息：exists/is_file/is_dir/is_symlink/is_absolute
│   ├── 非原地写入：临时文件 + replace() 防崩溃 ⭐⭐
│   └── vs os.path：pathlib首选 ⭐⭐⭐
│
├── JSON ⭐⭐
│   ├── dump/dumps/load/loads 四函数
│   ├── ⚠️ int键变str，tuple变list（类型丢失）
│   ├── JSONEncoder.default() → 自定义编码
│   └── object_hook → 自定义解码 ⭐
│
├── CSV → csv模块（reader/writer/DictReader/DictWriter）
└── INI → configparser模块
```

- **警告与提示表**

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 关键 | **必须关闭流**！不能依赖垃圾回收器；`with`语句是惯用方式     |
| ⚠️ 陷阱 | `w`/`w+`模式会**截断**文件内容，打开即清空！                 |
| ⚠️ 陷阱 | `x`/`x+`模式文件已存在时抛`FileExistsError`                  |
| ⚠️ 关键 | `write()`是**覆盖而非插入**，新内容短于旧内容需`truncate()`清理残留 |
| ⚠️ 关键 | `writelines()`**不会自动添加换行符**                         |
| ⚠️ 陷阱 | 相对路径基于**当前工作目录**（不是脚本所在目录）→ 用`Path(__file__).resolve().parent` |
| ⚠️ 陷阱 | `__exit__()`返回`True`会**抑制异常**（吞掉错误），谨慎使用   |
| ⚠️ 陷阱 | 生成器版上下文管理器**只能`yield`一次**，否则`RuntimeError`  |
| 💡 技巧 | 永远用`with`语句打开文件，自动关闭                           |
| 💡 技巧 | 只读一次时用流迭代`for line in file`，最Pythonic             |
| 💡 技巧 | `print()`+`file=`适合简单写入，利用自动换行和`sep=`          |
| 💡 技巧 | Python自动处理换行符差异，你永远只需用`\n`                   |
| 💡 技巧 | 重要文件用**临时文件+`replace()`**写入，防崩溃损坏           |
| 💡 技巧 | 简单清理逻辑用`@contextmanager`，复杂状态管理用类版本        |
| 💡 技巧 | `pathlib`新代码首选，路径拼接优先`/`运算符                   |